In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

membership_df = spark.table("segment_membership")
action_df = spark.table("action_catalog")
consent_df = spark.table("consent")
support_df = spark.table("support_case")

print("All tables loaded successfully!")

In [0]:
#Get the latest consent status
consent_lookup_df = (
    consent_df
    .select(
        "customer_id",
        "consent_status"
    )
)

display(consent_lookup_df)

In [0]:
#Identify customers with open support cases
open_support_df = (
    support_df
    .filter(F.col("case_status") == "OPEN")
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("open_case_count"),
        F.sum("service_cost").alias("total_service_cost")
    )
)

display(open_support_df)

In [0]:
#Join customer segments, consent and support information
customer_context_df = (
    membership_df
    .join(consent_lookup_df, "customer_id", "left")
    .join(open_support_df, "customer_id", "left")
    .withColumn(
        "consent_status",
        F.coalesce(F.col("consent_status"), F.lit("UNKNOWN"))
    )
    .withColumn(
        "open_case_count",
        F.coalesce(F.col("open_case_count"), F.lit(0))
    )
    .withColumn(
        "total_service_cost",
        F.coalesce(F.col("total_service_cost"), F.lit(0))
    )
)

display(customer_context_df)

In [0]:
#Select the recommended action
recommendation_df = (
    customer_context_df
    .withColumn(
        "recommended_action",
        F.when(
            F.col("open_case_count") > 0,
            "SUPPORT_FOLLOWUP"
        )
        .when(
            F.col("consent_status") != "GRANTED",
            "NO_MARKETING_ACTION"
        )
        .when(
            F.col("segment_name") == "Champions",
            "VIP_REWARD"
        )
        .when(
            F.col("segment_name") == "Loyal",
            "LOYALTY_OFFER"
        )
        .when(
            F.col("segment_name") == "New Customers",
            "WELCOME_OFFER"
        )
        .when(
            F.col("segment_name") == "At Risk",
            "RETENTION_OFFER"
        )
        .when(
            F.col("segment_name") == "Hibernating",
            "WINBACK_CAMPAIGN"
        )
        .otherwise("PRODUCT_RECOMMENDATION")
    )
    .withColumn(
        "recommendation_reason",
        F.when(
            F.col("open_case_count") > 0,
            "Customer has an open support case"
        )
        .when(
            F.col("consent_status") != "GRANTED",
            "Marketing consent is not granted"
        )
        .otherwise(
            F.concat(F.lit("Recommendation based on segment: "), F.col("segment_name"))
        )
    )
)

display(recommendation_df)

In [0]:
#Connect the recommendation to ActionCatalog
final_recommendation_df = (
    recommendation_df
    .join(
        action_df,
        recommendation_df.recommended_action == action_df.action_name,
        "left"
    )
    .withColumn(
        "recommendation_id",
        F.concat(
            F.lit("REC-2026-"),
            F.lpad(
                F.row_number().over(Window.orderBy("customer_id")).cast("string"),
                8,
                "0"
            )
        )
    )
    .withColumn(
        "recommendation_date",
        F.lit("2026-10-30").cast("date")
    )
    .select(
        "recommendation_id",
        "customer_id",
        "segment_name",
        "consent_status",
        "open_case_count",
        "recommended_action",
        "action_id",
        "action_type",
        "channel",
        "estimated_cost",
        "recommendation_reason",
        "recommendation_date"
    )
)

display(final_recommendation_df)

In [0]:
#Save the recommendations
final_recommendation_df.write.format("delta").mode("overwrite").saveAsTable(
    "customer_recommendation"
)

print("Customer recommendations created successfully!")

In [0]:
#Validate the recommendations
final_recommendation_df.groupBy(
    "recommended_action"
).count().orderBy("recommended_action").show()

In [0]:
final_recommendation_df.filter(
    (F.col("action_type") == "MARKETING") &
    (F.col("consent_status") != "GRANTED")
).show()

In [0]:
#Check customers with open cases:
final_recommendation_df.filter(
    F.col("open_case_count") > 0
).select(
    "customer_id",
    "recommended_action",
    "recommendation_reason"
).show()

In [0]:
#Check there is exactly one recommendation per customer
multi_recommendations = (
    final_recommendation_df
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") != 1)
)

print("Customers without exactly one recommendation:", multi_recommendations.count())

print(
    "Recommendations without a catalog action:",
    final_recommendation_df.filter(F.col("action_id").isNull()).count()
)